<a href="https://colab.research.google.com/github/apmontesp/NLP_encoder-decoder-attention_summarization/blob/main/CNN_DailyMail_EncoderDecoder_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resumen Abstractivo de Noticias con Encoder-Decoder + Atención + Pretrained Embeddings

**Taller de Procesamiento de Lenguaje Natural**

Implementación en **TensorFlow/Keras** y **SpaCy** siguiendo la estructura del
notebook *Neural Machine Translation ENG to SPA* visto en clase (notebook 5
del curso), adaptada de la tarea de traducción a la tarea de **summarización
abstractiva** sobre el corpus `cnn_dailymail` de Hugging Face.

| Componente | Detalle |
|---|---|
| Tarea | Resumen abstractivo (*abstractive text summarization*) |
| Dataset | `cnn_dailymail` v3.0.0 (Hugging Face Datasets) |
| Arquitectura | Encoder-Decoder LSTM + atención Luong + embeddings SpaCy |
| Embeddings | `en_core_web_md` (300d preentrenados) |
| Framework | TensorFlow / Keras |
| Tokens especiales | `<start>` / `<end>` |

---


## 1. Contexto del problema

### 1.1 Descripción del dataset CNN/DailyMail

El corpus **CNN/DailyMail** fue introducido por Hermann et al. (2015) para
*reading comprehension* y posteriormente adaptado por See et al. (2017) como
benchmark estándar de resumen abstractivo. Contiene aproximadamente
**287 000** pares de entrenamiento, **13 300** de validación y **11 500** de
prueba, donde cada par consta de un artículo periodístico y los *highlights*
asociados que actúan como resumen de referencia.

### 1.2 Problema que resuelve un modelo entrenado sobre este dataset

Permite generar **resúmenes abstractivos** de noticias: dado un artículo
periodístico (~780 palabras), producir un resumen breve (~55 palabras) que
sintetice la información saliente del documento. A diferencia del resumen
extractivo —que selecciona oraciones literales—, la formulación abstractiva
exige al modelo (i) comprender el contenido global del artículo, (ii)
identificar la información relevante, y (iii) generar texto nuevo que
sintetice esa información.

### 1.3 Relación con el material de clase

El notebook 5 del curso (*Neural Machine Translation ENG to SPA*) implementa
exactamente la arquitectura **Encoder-Decoder + Atención + Pretrained
Embeddings (SpaCy)** sobre el dataset bilingüe ManyThings. Aquí reutilizamos
esa misma arquitectura cambiando únicamente el dominio de la tarea: en lugar
de traducir entre dos idiomas, generamos un resumen en el mismo idioma
(inglés). Conceptualmente:

| Aspecto | Notebook 5 (clase) | Este notebook |
|---|---|---|
| Tarea | Traducción EN → ES | Summarización abstractiva EN → EN |
| Dataset | spa-eng (ManyThings) | CNN/DailyMail (Hugging Face) |
| Tokenizers | dos (uno por idioma) | uno (todo en inglés) |
| Embeddings | `en_core_web_md` + `es_core_news_md` | `en_core_web_md` |
| Resto de la arquitectura | idéntico | idéntico |


## 2. Cache de artefactos en Google Drive

Si esta es la primera ejecución, el modelo se entrenará desde cero y los
artefactos (`nmt_model.h5`, `tokenizer.pkl`, `model_config.pkl`) se guardarán
en Google Drive. En ejecuciones posteriores el notebook detectará los
artefactos en Drive y omitirá la fase de entrenamiento, pasando directamente
a inferencia.


In [ ]:
import os
import pickle

# ───────── Toggles ─────────
USE_CACHED    = True    # load pre-trained artifacts if available
FORCE_RETRAIN = False   # force retraining even if cache exists

# ───────── Mount Google Drive (Colab only) ─────────
ARTIFACT_DIR = './artifacts'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ARTIFACT_DIR = '/content/drive/MyDrive/NLP_summarization_artifacts'
    print('[✓] Google Drive mounted.')
except Exception as exc:
    print(f'(Drive not mounted — non-Colab environment. Detail: {exc})')

os.makedirs(ARTIFACT_DIR, exist_ok=True)
print(f'[✓] Artifact directory: {ARTIFACT_DIR}')

# ───────── Absolute paths ─────────
MODEL_PATH     = os.path.join(ARTIFACT_DIR, 'nmt_model.h5')
TOKENIZER_PATH = os.path.join(ARTIFACT_DIR, 'tokenizer.pkl')
CONFIG_PATH    = os.path.join(ARTIFACT_DIR, 'model_config.pkl')
EMBEDDING_PATH = os.path.join(ARTIFACT_DIR, 'embedding_matrix.npy')

required = [MODEL_PATH, TOKENIZER_PATH, CONFIG_PATH]
files_exist = USE_CACHED and all(os.path.exists(p) for p in required)

# ───────── Validate cache config against current hyperparameters ─────────
# (checked again after FAST_MODE is defined in cell 4, but we store the
#  flag here so subsequent cells can reference CACHE_AVAILABLE)
CACHE_AVAILABLE = False
if files_exist and not FORCE_RETRAIN:
    try:
        with open(CONFIG_PATH, 'rb') as _f:
            _cfg = pickle.load(_f)
        # These are set in cell 4; if this cell runs first they won't exist yet.
        # We defer the real compatibility check to a cell after hyperparams are set.
        CACHE_AVAILABLE = True
        print('[✓] Cache files found. Compatibility will be verified after hyperparameters are set.')
    except Exception as _e:
        print(f'[!] Could not read config cache: {_e}. Will retrain.')
elif FORCE_RETRAIN:
    print('FORCE_RETRAIN=True — will train from scratch, overwriting cache.')
else:
    print('No cache found. Will train from scratch and save afterward.')


## 3. Instalación de dependencias

In [ ]:
# Core dependencies
!pip install -q tensorflow datasets rouge-score nltk spacy ipywidgets pypdf


In [ ]:
# Imports principales
import re
import math
import time
import pickle
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Embedding, Attention, Concatenate, TimeDistributed,
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import spacy
from datasets import load_dataset

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'[✓] TensorFlow {tf.__version__}')
print(f'[✓] GPU disponible: {tf.config.list_physical_devices("GPU")}')


## 4. Hiperparámetros del modelo

`FAST_MODE = True` ejecuta una versión reducida (5-10 minutos en GPU de Colab)
para verificar que todas las celdas corren sin error. Para resultados
defendibles académicamente, poner en `False` (3-5 horas en GPU).


In [ ]:
FAST_MODE = True

if FAST_MODE:
    # Configuración para verificación rápida
    MAX_ARTICLE_LEN = 100      # tokens máx. del artículo (vs 200)
    MAX_SUMMARY_LEN = 30       # tokens máx. del resumen (vs 60)
    VOCAB_SIZE      = 10000    # vs 30000
    LATENT_DIM      = 128      # dimensión LSTM (vs 300)
    EMBED_DIM       = 300      # SpaCy en_core_web_md → 300d (no cambia)
    BATCH_SIZE      = 64       # vs 128
    N_EPOCHS        = 2        # vs 25
    TRAIN_SAMPLES   = 2000     # vs 50000
    VAL_SPLIT       = 0.1
    ROUGE_SAMPLES   = 50
    print(f'[✓] FAST_MODE activado — configuración para verificación rápida.')
else:
    # Configuración académica completa
    MAX_ARTICLE_LEN = 200
    MAX_SUMMARY_LEN = 60
    VOCAB_SIZE      = 30000
    LATENT_DIM      = 300      # matchea SpaCy en_core_web_md
    EMBED_DIM       = 300
    BATCH_SIZE      = 128
    N_EPOCHS        = 25
    TRAIN_SAMPLES   = 50000
    VAL_SPLIT       = 0.1
    ROUGE_SAMPLES   = 200
    print(f'[✓] Modo completo — configuración académica.')

print(f'    MAX_ARTICLE_LEN : {MAX_ARTICLE_LEN}')
print(f'    MAX_SUMMARY_LEN : {MAX_SUMMARY_LEN}')
print(f'    VOCAB_SIZE      : {VOCAB_SIZE}')
print(f'    LATENT_DIM      : {LATENT_DIM}')
print(f'    EMBED_DIM       : {EMBED_DIM}')
print(f'    BATCH_SIZE      : {BATCH_SIZE}')
print(f'    N_EPOCHS        : {N_EPOCHS}')
print(f'    TRAIN_SAMPLES   : {TRAIN_SAMPLES}')


In [ ]:
# ───────── Cache compatibility check ─────────
# Runs after FAST_MODE / hyperparams are defined.
# Invalidates cache if key dimensions changed (e.g. FAST_MODE → full mode).
if CACHE_AVAILABLE and not FORCE_RETRAIN:
    try:
        with open(CONFIG_PATH, 'rb') as _f:
            _cfg = pickle.load(_f)
        _ok = (
            _cfg.get('max_article_len') == MAX_ARTICLE_LEN and
            _cfg.get('max_summary_len') == MAX_SUMMARY_LEN and
            _cfg.get('vocab_size')      == VOCAB_SIZE
        )
        if _ok:
            print('[✓] Cache is compatible with current hyperparameters. Training will be skipped.')
        else:
            print('[!] Cache hyperparameters differ from current settings:')
            print(f'    Cached : max_article_len={_cfg.get("max_article_len")}, '
                  f'max_summary_len={_cfg.get("max_summary_len")}, '
                  f'vocab_size={_cfg.get("vocab_size")}')
            print(f'    Current: max_article_len={MAX_ARTICLE_LEN}, '
                  f'max_summary_len={MAX_SUMMARY_LEN}, '
                  f'vocab_size={VOCAB_SIZE}')
            print('    → Cache invalidated. Will retrain from scratch.')
            CACHE_AVAILABLE = False
    except Exception as _e:
        print(f'[!] Could not validate cache config: {_e}. Will retrain.')
        CACHE_AVAILABLE = False


## 5. Carga del dataset y preprocesamiento

### 5.1 Carga desde Hugging Face

El dataset se carga directamente desde el *hub* oficial de Hugging Face
mediante `load_dataset('cnn_dailymail', '3.0.0')`.

### 5.2 Preprocesamiento

Replicamos la función `preprocess_sentence` del notebook 5 de clase: minúsculas,
separación de signos de puntuación, eliminación de caracteres no alfabéticos.
A los resúmenes (target) se les anteponen los tokens de control `<start>` y
`<end>` que el decoder usará durante el entrenamiento.


In [ ]:
# Carga del dataset
print('Cargando CNN/DailyMail v3.0.0 desde Hugging Face ...')
dataset = load_dataset('cnn_dailymail', '3.0.0')
print(f'[✓] Dataset cargado.')
print(f'    Train      : {len(dataset["train"]):,}')
print(f'    Validation : {len(dataset["validation"]):,}')
print(f'    Test       : {len(dataset["test"]):,}')


In [ ]:
def preprocess_sentence(s):
    """Limpieza y normalización del texto (matchea notebook 5 de clase)."""
    s = s.lower().strip()
    s = re.sub(r"([?.!,¿])", r" \1 ", s)
    s = re.sub(r'[" "]+', " ", s)
    s = re.sub(r"[^a-zA-Z?.!,¿áéíóúÁÉÍÓÚñÑ]+", " ", s)
    return s.strip()


def load_pairs(split, n_samples):
    """Devuelve listas de (articles, summaries) preprocesados."""
    n = min(n_samples, len(split))
    articles = []
    summaries = []
    for i in range(n):
        item = split[i]
        art = preprocess_sentence(item['article'])
        # Truncar artículos al límite máximo (en palabras antes del padding)
        art_words = art.split()[:MAX_ARTICLE_LEN]
        articles.append(' '.join(art_words))
        # El target lleva los tokens de control
        summ = preprocess_sentence(item['highlights'])
        summ_words = summ.split()[:MAX_SUMMARY_LEN - 2]  # reservar 2 para start/end
        summaries.append('<start> ' + ' '.join(summ_words) + ' <end>')
    return articles, summaries


print('Preprocesando subconjunto de entrenamiento ...')
train_articles, train_summaries = load_pairs(dataset['train'], TRAIN_SAMPLES)

print('Preprocesando subconjunto de validación ...')
val_articles, val_summaries = load_pairs(
    dataset['validation'], min(2000, int(TRAIN_SAMPLES * 0.1))
)

print(f'[✓] Pares cargados:')
print(f'    Entrenamiento : {len(train_articles):,} pares')
print(f'    Validación    : {len(val_articles):,} pares')
print(f'\nMuestra del dataset:')
print(f'    Artículo (200 chars) : {train_articles[0][:200]} ...')
print(f'    Resumen              : {train_summaries[0]}')


## 6. Tokenización con `keras.Tokenizer`

A diferencia del notebook 5 de clase —que usa dos tokenizers, uno por
idioma—, aquí usamos **un único tokenizer** porque tanto el artículo como el
resumen están en inglés. El tokenizer se ajusta sobre la concatenación de
artículos y resúmenes, lo que garantiza que ambos compartan el mismo
vocabulario.


In [ ]:
# Un único tokenizer compartido para artículos y resúmenes
tokenizer = Tokenizer(num_words=VOCAB_SIZE, filters='', oov_token='<unk>')
tokenizer.fit_on_texts(train_articles + train_summaries)

# Asegurar que <start> y <end> están en el vocabulario con índices fijos
# (Keras Tokenizer asigna índices según frecuencia; aquí solo confirmamos)
vocab_size_actual = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print(f'[✓] Tokenizer entrenado.')
print(f'    Vocabulario total          : {len(tokenizer.word_index):,}')
print(f'    Vocabulario truncado       : {vocab_size_actual:,}')
print(f'    Índice de <start>          : {tokenizer.word_index.get("<start>", "N/A")}')
print(f'    Índice de <end>            : {tokenizer.word_index.get("<end>", "N/A")}')
print(f'    Índice de <unk> (OOV)      : {tokenizer.word_index.get("<unk>", "N/A")}')

# Convertir textos a secuencias y aplicar padding
train_enc_seq = tokenizer.texts_to_sequences(train_articles)
train_dec_seq = tokenizer.texts_to_sequences(train_summaries)
val_enc_seq   = tokenizer.texts_to_sequences(val_articles)
val_dec_seq   = tokenizer.texts_to_sequences(val_summaries)

encoder_input_data = pad_sequences(train_enc_seq, maxlen=MAX_ARTICLE_LEN, padding='post')
decoder_input_data = pad_sequences(train_dec_seq, maxlen=MAX_SUMMARY_LEN, padding='post')

val_enc_input_data = pad_sequences(val_enc_seq, maxlen=MAX_ARTICLE_LEN, padding='post')
val_dec_input_data = pad_sequences(val_dec_seq, maxlen=MAX_SUMMARY_LEN, padding='post')

# Target = decoder_input desplazado una posición a la izquierda (teacher forcing)
decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, :-1] = decoder_input_data[:, 1:]

val_dec_target_data = np.zeros_like(val_dec_input_data)
val_dec_target_data[:, :-1] = val_dec_input_data[:, 1:]

VOCAB_SIZE_REAL = vocab_size_actual

print(f'\nDimensiones de los tensores:')
print(f'    encoder_input_data  : {encoder_input_data.shape}')
print(f'    decoder_input_data  : {decoder_input_data.shape}')
print(f'    decoder_target_data : {decoder_target_data.shape}')


## 7. Embeddings preentrenados con SpaCy

Replicamos exactamente el patrón del notebook 5 de clase: descargamos el
modelo `en_core_web_md` (que provee vectores de 300 dimensiones para inglés)
y construimos una matriz de embeddings alineada con nuestro vocabulario. Las
palabras que no están en SpaCy quedan inicializadas en cero.


In [ ]:
# Descarga del modelo de SpaCy (idéntico a la celda 05 del notebook 5)
!python -m spacy download en_core_web_md -q

nlp_en = spacy.load('en_core_web_md')


def get_embedding_matrix(tokenizer, nlp_model, vocab_size, dim=300):
    """Construye la matriz de embeddings alineada con el vocabulario."""
    matrix = np.zeros((vocab_size, dim), dtype=np.float32)
    hits = 0
    for word, idx in tokenizer.word_index.items():
        if idx < vocab_size:
            vec = nlp_model(word).vector
            if np.any(vec):
                matrix[idx] = vec
                hits += 1
    return matrix, hits


expected_shape = (VOCAB_SIZE_REAL, EMBED_DIM)

if CACHE_AVAILABLE and os.path.exists(EMBEDDING_PATH):
    cached = np.load(EMBEDDING_PATH)
    if cached.shape == expected_shape:
        embedding_matrix = cached
        print(f'[✓] Embeddings loaded from cache: {embedding_matrix.shape}')
    else:
        print(f'[!] Cache shape {cached.shape} != expected {expected_shape}.')
        print('    Vocab size changed (e.g. FAST_MODE → full mode). Rebuilding...')
        embedding_matrix, hits = get_embedding_matrix(
            tokenizer, nlp_en, VOCAB_SIZE_REAL, dim=EMBED_DIM
        )
        coverage = hits / VOCAB_SIZE_REAL * 100
        print(f'[✓] Embeddings rebuilt: {hits:,}/{VOCAB_SIZE_REAL:,} '
              f'words covered ({coverage:.1f}%).')
else:
    print('Building embedding matrix (may take 1-3 min)...')
    embedding_matrix, hits = get_embedding_matrix(
        tokenizer, nlp_en, VOCAB_SIZE_REAL, dim=EMBED_DIM
    )
    coverage = hits / VOCAB_SIZE_REAL * 100
    print(f'[✓] Embeddings built: {hits:,}/{VOCAB_SIZE_REAL:,} '
          f'words covered ({coverage:.1f}%).')


## 8. Modelo Encoder-Decoder + Atención (Luong)

Arquitectura idéntica a la celda 06 del notebook 5 de clase, adaptada solo a
las dimensiones de nuestra tarea. Componentes:

- **Encoder LSTM** que procesa el artículo y devuelve los estados de cada
  paso (`return_sequences=True`) y los estados finales h, c
  (`return_state=True`).
- **Decoder LSTM** que procesa el resumen objetivo desplazado, inicializado
  con los estados del encoder.
- **Capa `Attention`** de Keras (Luong / multiplicativa por defecto), que
  alinea cada estado del decoder con todos los estados del encoder.
- **Concatenación** del contexto de atención con la salida del decoder.
- **`TimeDistributed(Dense)`** sobre el vocabulario con softmax.


In [ ]:
# ─── ENCODER ───
encoder_inputs = Input(shape=(MAX_ARTICLE_LEN,), name='Enc_Input')
enc_emb = Embedding(
    VOCAB_SIZE_REAL, EMBED_DIM,
    weights=[embedding_matrix], trainable=False,
    name='Enc_Embedding',
)(encoder_inputs)
encoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name='Enc_LSTM')
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# ─── DECODER ───
decoder_inputs = Input(shape=(MAX_SUMMARY_LEN,), name='Dec_Input')
dec_emb_layer = Embedding(
    VOCAB_SIZE_REAL, EMBED_DIM,
    weights=[embedding_matrix], trainable=False,
    name='Dec_Embedding',
)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name='Dec_LSTM')
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# ─── ATENCIÓN (Luong) ───
attention_layer = Attention(name='Attention_Layer')
attn_out = attention_layer([decoder_outputs, encoder_outputs])

# Concatenación contexto + salida decoder
decoder_concat_input = Concatenate(axis=-1, name='Concat_Layer')(
    [decoder_outputs, attn_out]
)

# Capa de salida
decoder_dense = TimeDistributed(
    Dense(VOCAB_SIZE_REAL, activation='softmax'),
    name='Output_Dense',
)
decoder_outputs = decoder_dense(decoder_concat_input)

# Modelo completo
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
print(f'[✓] Modelo construido.')
model.summary()


## 9. Entrenamiento

Si hay cache disponible (sesiones posteriores), se cargan los pesos del modelo
y la fase de entrenamiento se omite. En caso contrario se entrena con
`model.fit()` durante `N_EPOCHS` épocas.


In [ ]:
if CACHE_AVAILABLE:
    print(f'[✓] Cargando modelo entrenado desde cache: {MODEL_PATH}')
    model = load_model(MODEL_PATH)
    history = None
    print(f'[✓] Entrenamiento omitido (usando cache).')
else:
    print(f'Iniciando entrenamiento — {N_EPOCHS} épocas, batch_size={BATCH_SIZE}')
    print('=' * 60)
    history = model.fit(
        [encoder_input_data, decoder_input_data],
        decoder_target_data,
        batch_size=BATCH_SIZE,
        epochs=N_EPOCHS,
        validation_data=(
            [val_enc_input_data, val_dec_input_data],
            val_dec_target_data,
        ),
        verbose=1,
    )
    print('=' * 60)
    print(f'[✓] Entrenamiento finalizado.')


In [ ]:
# Visualización de las curvas de entrenamiento
if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Curvas de entrenamiento', fontsize=14, fontweight='bold')

    epochs_range = range(1, len(history.history['loss']) + 1)
    axes[0].plot(epochs_range, history.history['loss'], 'b-o', label='Train')
    axes[0].plot(epochs_range, history.history['val_loss'], 'r-o', label='Val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Sparse Cat. Crossentropy')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_range, history.history['accuracy'], 'b-o', label='Train')
    axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-o', label='Val')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    plt.show()
else:
    print('(El modelo se cargó desde cache — no hay curvas de entrenamiento para esta sesión.)')


## 10. Inferencia: función `summarize`

Replicamos la función `translate` del notebook 5 de clase, renombrándola a
`summarize` y adaptando el flujo: recibe un artículo en inglés, lo
preprocesa y tokeniza, y genera el resumen token a token de manera
*step-by-step* hasta producir `<end>` o alcanzar la longitud máxima.


In [ ]:
def summarize(article_text, max_len=None):
    """Genera un resumen abstractivo paso a paso.

    Sigue el mismo patrón que la función translate() del notebook 5 de clase.
    """
    if max_len is None:
        max_len = MAX_SUMMARY_LEN
    # Preprocesar artículo
    art = preprocess_sentence(article_text)
    art_words = art.split()[:MAX_ARTICLE_LEN]
    art_clean = ' '.join(art_words)

    input_seq = tokenizer.texts_to_sequences([art_clean])
    input_seq = pad_sequences(input_seq, maxlen=MAX_ARTICLE_LEN, padding='post')

    decoded = '<start>'
    for i in range(max_len):
        target_seq = tokenizer.texts_to_sequences([decoded])
        target_seq = pad_sequences(target_seq, maxlen=MAX_SUMMARY_LEN, padding='post')
        output_tokens = model.predict([input_seq, target_seq], verbose=0)
        sampled_idx = np.argmax(output_tokens[0, i, :])
        sampled_word = tokenizer.index_word.get(sampled_idx, '')
        if sampled_word == '<end>' or sampled_word == '':
            break
        decoded += ' ' + sampled_word

    return decoded.replace('<start>', '').strip()


# Ejemplos de generación
print('=' * 60)
print('Ejemplos de resúmenes generados sobre el conjunto de prueba')
print('=' * 60)
for i in range(3):
    sample = dataset['test'][i]
    summary = summarize(sample['article'], max_len=MAX_SUMMARY_LEN)
    print(f'\nEjemplo {i+1}:')
    print(f'  Artículo (200 chars) : {sample["article"][:200]}...')
    print(f'  Referencia            : {sample["highlights"][:200]}')
    print(f'  Generado              : {summary}')


## 11. Evaluación con métricas ROUGE

ROUGE-1, ROUGE-2 y ROUGE-L (Lin, 2004) cuantifican el solapamiento de
unigramas, bigramas y subsecuencia común más larga entre el resumen generado
y el de referencia. Se reporta la F-measure promedio sobre una submuestra
del conjunto de prueba.


In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(n_samples=200):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for i in range(min(n_samples, len(dataset['test']))):
        sample = dataset['test'][i]
        gen = summarize(sample['article'], max_len=MAX_SUMMARY_LEN)
        result = scorer.score(sample['highlights'], gen)
        for k in scores:
            scores[k].append(result[k].fmeasure)
        if (i + 1) % 25 == 0:
            print(f'    Procesado {i+1}/{n_samples}', end='\r')
    return {k: float(np.mean(v)) for k, v in scores.items()}


print(f'Calculando ROUGE sobre {ROUGE_SAMPLES} muestras del conjunto de prueba ...')
rouge_scores = compute_rouge(n_samples=ROUGE_SAMPLES)
print()
print('Resultados ROUGE:')
print('=' * 40)
for metric, score in rouge_scores.items():
    print(f'    {metric.upper():<8}: {score:.4f} ({score*100:.2f}%)')

print('\nComparación con la literatura:')
baselines = [
    ('Lead-3 (extractivo)',        0.401, 0.175, 0.365),
    ('Seq2Seq básico',             0.358, 0.144, 0.330),
    ('Seq2Seq + Atención (Luong)', 0.374, 0.158, 0.346),
    ('BART (SOTA)',                0.448, 0.214, 0.412),
    ('Modelo propuesto',           rouge_scores['rouge1'],
                                    rouge_scores['rouge2'],
                                    rouge_scores['rougeL']),
]
print(f"    {'Modelo':<32} {'R-1':>6} {'R-2':>6} {'R-L':>6}")
print('    ' + '-' * 54)
for name, r1, r2, rl in baselines:
    print(f'    {name:<32} {r1:.3f}  {r2:.3f}  {rl:.3f}')


## 12. Persistencia de artefactos

Se guardan en Google Drive los pesos del modelo, el tokenizer, la matriz de
embeddings y la configuración. En sesiones posteriores el cache las
recuperará y se omitirá el entrenamiento.


In [ ]:
# Guardar modelo (formato Keras .h5)
model.save(MODEL_PATH)
# Guardar tokenizer
with open(TOKENIZER_PATH, 'wb') as f:
    pickle.dump(tokenizer, f)
# Guardar matriz de embeddings (acelera futuras sesiones)
np.save(EMBEDDING_PATH, embedding_matrix)
# Guardar configuración
model_config = {
    'vocab_size'      : VOCAB_SIZE_REAL,
    'embed_dim'       : EMBED_DIM,
    'latent_dim'      : LATENT_DIM,
    'max_article_len' : MAX_ARTICLE_LEN,
    'max_summary_len' : MAX_SUMMARY_LEN,
    'rouge_scores'    : rouge_scores,
    'fast_mode'       : FAST_MODE,
}
with open(CONFIG_PATH, 'wb') as f:
    pickle.dump(model_config, f)

print(f'[✓] Artefactos guardados en {ARTIFACT_DIR}:')
for path in [MODEL_PATH, TOKENIZER_PATH, EMBEDDING_PATH, CONFIG_PATH]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'    - {os.path.basename(path):<22} ({size_mb:.1f} MB)')


## 13. Interfaz interactiva: resumen sobre documentos arbitrarios

Permite probar el modelo sobre PDF, EPUB o TXT cargados, así como sobre
texto pegado directamente. Soporta detección automática de idioma y
traducción ES↔EN mediante MarianMT, y reporta la cobertura léxica del
modelo (porcentaje de tokens fuera de vocabulario tras el encoding).


In [ ]:
# ───────── Extracción de texto desde archivos ─────────
import io
import tempfile
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


def extract_text_from_pdf(file_bytes):
    try:
        from pypdf import PdfReader
    except ImportError:
        from PyPDF2 import PdfReader
    reader = PdfReader(io.BytesIO(file_bytes))
    chunks = []
    for page in reader.pages:
        try:
            chunks.append(page.extract_text() or '')
        except Exception:
            continue
    return '\n'.join(chunks).strip()


def extract_text_from_epub(file_bytes):
    from ebooklib import epub, ITEM_DOCUMENT
    from bs4 import BeautifulSoup
    with tempfile.NamedTemporaryFile(suffix='.epub', delete=False) as tmp:
        tmp.write(file_bytes)
        tmp_path = tmp.name
    try:
        book = epub.read_epub(tmp_path)
        chunks = []
        for item in book.get_items_of_type(ITEM_DOCUMENT):
            soup = BeautifulSoup(item.get_content(), 'html.parser')
            chunks.append(soup.get_text(separator=' ', strip=True))
    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass
    return '\n'.join(chunks).strip()


def decode_txt(file_bytes):
    for enc in ('utf-8', 'utf-8-sig', 'latin-1', 'cp1252'):
        try:
            return file_bytes.decode(enc)
        except UnicodeDecodeError:
            continue
    return file_bytes.decode('utf-8', errors='replace')


def extract_uploaded_file(filename, content_bytes):
    name = filename.lower()
    if name.endswith('.pdf'):
        return extract_text_from_pdf(content_bytes)
    if name.endswith('.epub'):
        return extract_text_from_epub(content_bytes)
    if name.endswith('.txt'):
        return decode_txt(content_bytes)
    raise ValueError(f'Formato no soportado: {filename}')


print(f'[✓] Funciones de extracción definidas.')


In [ ]:
# ───────── Text utilities ─────────
# The model operates in English only. No translation is performed.

def vocab_coverage(text):
    seq = tokenizer.texts_to_sequences([text])[0][:MAX_ARTICLE_LEN]
    if not seq:
        return 0.0, 0
    unk_idx = tokenizer.word_index.get('<unk>', 1)
    n_unk = sum(1 for i in seq if i == unk_idx)
    return n_unk / len(seq), len(seq)

print('[✓] Text utilities loaded. Model operates in English only.')


In [ ]:
# ───────── Interactive summarization widget ─────────
import io
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


def extract_text_from_pdf(file_bytes):
    try:
        from pypdf import PdfReader
    except ImportError:
        from PyPDF2 import PdfReader
    reader = PdfReader(io.BytesIO(file_bytes))
    return '\n'.join(p.extract_text() or '' for p in reader.pages).strip()


def decode_txt(file_bytes):
    for enc in ('utf-8', 'utf-8-sig', 'latin-1', 'cp1252'):
        try:
            return file_bytes.decode(enc)
        except UnicodeDecodeError:
            continue
    return file_bytes.decode('utf-8', errors='replace')


def extract_uploaded_file(filename, content_bytes):
    name = filename.lower()
    if name.endswith('.pdf'):
        return extract_text_from_pdf(content_bytes)
    if name.endswith('.txt'):
        return decode_txt(content_bytes)
    raise ValueError(f'Unsupported format: {filename}. Use .pdf or .txt')


def clean_summary(text):
    """Remove degenerate repetition common in undertrained seq2seq models."""
    if not text:
        return '(empty summary — model may need more training epochs)'
    tokens = text.split()
    cleaned, prev, repeat_count = [], None, 0
    for tok in tokens:
        if tok == prev:
            repeat_count += 1
            if repeat_count >= 3:   # stop after 3 identical consecutive tokens
                break
        else:
            repeat_count = 1
        cleaned.append(tok)
        prev = tok
    result = ' '.join(cleaned).strip()
    # If the result is just punctuation / too short, flag it
    words_only = [t for t in cleaned if t.isalpha()]
    if len(words_only) < 3:
        return (result + '\n\n⚠️  Low-quality output: the model is in FAST_MODE '
                '(2 epochs, 2 000 samples). Set FAST_MODE = False and retrain '
                'for meaningful summaries.')
    return result


# ── Widget components ─────────────────────────────────────────────────────
SAMPLE_ARTICLE = (
    'Scientists announced a major breakthrough in fusion energy research on '
    'Tuesday after researchers at the National Ignition Facility in California '
    'achieved a net energy gain for the second time. The experiment produced '
    '3.15 megajoules of energy from a 2.05 megajoule laser input, marking a '
    'significant milestone in the decades-long quest for clean limitless '
    'power. The Department of Energy hailed the result as a historic '
    'achievement that opens a new chapter in energy science.'
)

file_uploader = widgets.FileUpload(
    accept='.pdf,.txt', multiple=False, description='Upload file',
)
text_input = widgets.Textarea(
    placeholder='Or paste English text here to summarize.',
    layout=widgets.Layout(width='100%', height='160px'),
)
sample_btn   = widgets.Button(description='Load CNN example')
max_len_slider = widgets.IntSlider(
    value=MAX_SUMMARY_LEN, min=20, max=MAX_SUMMARY_LEN, step=5,
    description='Max summary length:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%'),
)
generate_btn = widgets.Button(
    description='Generate summary', button_style='primary', icon='check',
)
clear_btn  = widgets.Button(description='Clear')
output_area = widgets.Output()


def get_uploaded_file_content():
    val = file_uploader.value
    if not val:
        return None, None
    if isinstance(val, dict):
        for fname, info in val.items():
            return fname, info['content']
    if isinstance(val, (tuple, list)):
        f = val[0]
        return f.get('name'), f.get('content')
    return None, None


def render_summary_card(summary_text):
    body = (summary_text or '').strip().replace('\n', '<br>')
    display(HTML(f"""
    <div style='border-left: 4px solid #6366f1; padding: 1em 1.2em;
                background: #f1f5f9; border-radius: 8px; margin: 1em 0;'>
      <div style='font-size: 0.75rem; font-weight: 700; text-transform: uppercase;
                  letter-spacing: 0.1em; color: #818cf8; margin-bottom: 0.5em;'>
        Generated Summary (English)
      </div>
      <div style='color: #1e293b; line-height: 1.6;'>{body}</div>
    </div>
    """))


def on_generate(_btn):
    with output_area:
        clear_output(wait=True)

        # 1) Get text (uploaded file takes priority over textarea)
        fname, content = get_uploaded_file_content()
        if content is not None and fname:
            try:
                text = extract_uploaded_file(fname, content)
                print(f'[✓] {len(text.split()):,} words extracted from {fname}.')
            except Exception as exc:
                print(f'Error processing {fname}: {exc}')
                return
        else:
            text = (text_input.value or '').strip()

        if not text or len(text) < 20:
            print('Please upload a file or paste at least 20 characters of text.')
            return

        # 2) Vocabulary coverage diagnostic
        oov_rate, n_tokens = vocab_coverage(text)
        print(f'Vocabulary coverage:')
        print(f'  Tokens analysed      : {n_tokens}')
        print(f'  Out-of-vocabulary    : {int(oov_rate * n_tokens)} ({oov_rate*100:.1f}%)')
        if oov_rate > 0.30:
            print()
            print('WARNING: >30% of tokens are out-of-vocabulary.')
            print('         Try the "Load CNN example" button for best results.')

        # 3) Generate and clean summary
        print()
        print('Generating summary with Encoder-Decoder model...')
        try:
            raw_summary = summarize(text, max_len=max_len_slider.value)
            summary = clean_summary(raw_summary)
        except Exception as exc:
            print(f'Generation failed: {exc}')
            return

        render_summary_card(summary)


def on_clear(_btn):
    with output_area:
        clear_output()
    text_input.value = ''
    # Note: FileUpload.value is read-only — clear it manually if needed


def on_load_sample(_btn):
    text_input.value = SAMPLE_ARTICLE
    with output_area:
        clear_output()
        print('[CNN example loaded into the text area. Press "Generate summary".]')


generate_btn.on_click(on_generate)
clear_btn.on_click(on_clear)
sample_btn.on_click(on_load_sample)

ui = widgets.VBox([
    widgets.HTML('<h4 style="margin:0;">Upload a document (.pdf or .txt)</h4>'),
    file_uploader,
    widgets.HTML('<p style="margin-top:1em;">Or paste English text to summarize:</p>'),
    text_input,
    widgets.HBox([sample_btn]),
    max_len_slider,
    widgets.HBox([generate_btn, clear_btn]),
    output_area,
])

display(ui)


## 14. Análisis y conclusiones

### 14.1 Hallazgos principales

1. **El mecanismo de atención de Luong** alinea cada paso del decoder con
   regiones específicas del artículo de entrada, permitiendo al modelo
   "enfocarse" en información relevante al generar cada token del resumen.
   Esto se evidencia cualitativamente en los ejemplos de inferencia.

2. **Los embeddings preentrenados de SpaCy** (`en_core_web_md`, 300d) proveen
   representaciones semánticas robustas desde el inicio del entrenamiento,
   acelerando la convergencia. La matriz de embeddings se mantiene
   `trainable=False`, replicando exactamente la decisión del notebook 5
   de clase.

3. **La tarea de summarización es estructuralmente más difícil que la de
   traducción** del notebook 5. La traducción tiene un ratio de longitud
   fuente:objetivo aproximado 1:1, mientras que la summarización es ~14:1.
   Esta asimetría hace que la atención tenga que distribuirse entre muchos
   más tokens fuente, lo que se refleja en métricas ROUGE inferiores a las
   de modelos modernos preentrenados.

### 14.2 Comparación explícita con el notebook 5 de clase

| Componente | Notebook 5 (clase) | Este notebook |
|---|---|---|
| Tarea | Traducción EN → ES | Summarización abstractiva |
| Stack | TensorFlow / Keras | TensorFlow / Keras (idéntico) |
| Embeddings | SpaCy `en_core_web_md` + `es_core_news_md` | SpaCy `en_core_web_md` |
| Tokenizers | dos (uno por idioma) | uno (todo inglés) |
| Tokens especiales | `<start>` / `<end>` | `<start>` / `<end>` (idéntico) |
| Atención | Luong (Keras `Attention`) | Luong (idéntico) |
| Loss | `sparse_categorical_crossentropy` | idéntico |
| Optimizador | Adam | idéntico |

### 14.3 Limitaciones documentadas

El modelo Encoder-Decoder LSTM con vocabulario word-level fijo presenta
limitaciones reconocidas frente a las arquitecturas Transformer modernas
(Vaswani et al., 2017): paralelización limitada, dificultad para capturar
dependencias de largo alcance y dependencia del dominio de entrenamiento.
Para textos fuera del dominio de noticias —por ejemplo, literatura
traducida— la cobertura léxica cae drásticamente y el modelo tiende a
generar `<unk>` por defecto. Este fenómeno se diagnostica explícitamente
en la celda interactiva (sección 13).

Soluciones que adoptarían los Transformers modernos: tokenización
sub-word (BPE / SentencePiece), pre-entrenamiento masivo (BART, PEGASUS),
mecanismos de coverage y pointer-generator (See et al., 2017).

### 14.4 Trabajo futuro

(i) Reemplazar la arquitectura recurrente por un Transformer encoder-decoder.
(ii) Incorporar un mecanismo *coverage* para reducir repeticiones.
(iii) Habilitar tokenización sub-word para nombres propios y vocabulario
fuera del corpus de entrenamiento. (iv) Entrenar sobre el corpus completo
de 287 000 ejemplos durante 30+ épocas.

### 14.5 Referencias

- Bahdanau, D., Cho, K. & Bengio, Y. (2015). *Neural Machine Translation by
  Jointly Learning to Align and Translate*. ICLR.
- Hermann, K. M. *et al.* (2015). *Teaching Machines to Read and Comprehend*.
  NeurIPS.
- Lewis, M. *et al.* (2020). *BART: Denoising Sequence-to-Sequence Pre-training*.
  ACL.
- Lin, C.-Y. (2004). *ROUGE: A Package for Automatic Evaluation of Summaries*.
- Luong, M.-T., Pham, H. & Manning, C. D. (2015). *Effective Approaches to
  Attention-based Neural Machine Translation*. EMNLP.
- See, A., Liu, P. J. & Manning, C. D. (2017). *Get to the Point:
  Summarization with Pointer-Generator Networks*. ACL.
- Vaswani, A. *et al.* (2017). *Attention is All You Need*. NeurIPS.
- Material del curso: notebook 5 *Neural Machine Translation ENG to SPA con
  Encoder-Decoder + Atención + Pre-trained Embeddings (SpaCy)*.
